## Task 1

Here i should answer questions

> What is leave-one-out? Provide limitations and strengths.

Leave-One-out cross validation is a special case of k-fold cross-validation where k=number of data points.  and in each iteration, one observation is used for testing, and the rest are used for training. This process repeats n times where n means total number of samples.

the limitations;

<< computationally expensive

<< high variance: the training sets are very similar, so predictions can vary widely depending on the left-out sample

<< not suitable for large datasets

the strengths:

<< maximally efficient use of data meaning that each sample is tested once

<< no randomness meaning deterministic outcome

<< ideal when data is little

> How do Grid Search, Randomized Grid Search, and Bayesian optimization work?

grid search:

* tries all possible combinations of hyperparameters from a specified list
* exhaustive but slow, espicially with many hyperparameters or values
* guarantees to find the best rersult if within the grid

randomized grid search:

* randomly samples combinations from the hyperparameter space
* not exhaustive meaning faster and more efficient
* good for large hyperparameter spaces

bayesian optimization:

* builds a probabilistic model of the function that maps hyperparameters to performance
* chooses the next hyperparameters based on previous results = smart search
* efficient and faster convergence to the best result with fewer iterations


> Explain classification of feature selection methods. Explain how Pearson and Chi2 work. Explain how Lasso works. Explain what permutation significance is. Become familiar with SHAP.

feature selection methods - classification

feature selection techniques fall into 3 categories;

1) filter methods
* based on statistical tests
* independent of the ml model
* examples: pearson correlation and chi2

2) wrapper methods
* use the model's performance as a guide
* train/test model with different subsets of features
* examples: recursive feature elimination

3) embedded methods
* feature selection is part of the model training
* examples: lasso(l1 regularization), tree-based feature importance

how pearson and chi2 work

pearson (for numerical features):
* measures linear correlation between each feature and the target
* closer to 1 or -1 means strong linear relation
* only works well for continuous data and numerical targets

chi2 (for categorical features):
* tests independence between a feature and the target
* works with categorical features and binary/multiclass targets
* the higher the chi2 statistic, the more dependent and likely relevant the feature is

how lasso works

lasso(least absolute shrinkage and selection operator):
* a linear model with L1 regularization
* adds penalty: λ * sum(abs(coefficients)) to the loss function
* l1 regularization encourages sparsity, it sets irrelevant feature coefficients to exactly zero
* helps in feature selection + reducing overfitting

what is rermutation significance?

* a model-agnostic method for evaluating feature importance
* steps:
1. measure model accuracy on validation set
2. randomly shuffle one feature column
3. measure drop in accuracy
4. repeat for each feature
* features that cause large accuracy drop are more important
* good for non-linear models

SHAP(shapley additive explainations)
* based on game theory
* explains how much each feature contributes to a single prediction
* works for any model
* gives:
1) global feature importance
2) local interpretability (why a particular prediction happened)
* visual tools: SHAP summary plots, waterfall plots, dependence plots




## Task 2

imports

In [1]:
!pip install optuna
!pip install shap
!pip install ipywidgets

In [2]:

import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import Lasso, ElasticNet, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, log_loss, accuracy_score, make_scorer
import numpy as np
import shap
import time
from sklearn.inspection import permutation_importance
import optuna
import warnings
warnings.filterwarnings('ignore')

now im loading the data

In [3]:
train_df = pd.read_json('train.json')
test_df = pd.read_json('test.json')

print("Original Train shape:", train_df.shape)
print("Original Test shape:", test_df.shape)

Original Train shape: (49352, 15)
Original Test shape: (74659, 14)


Im initializing the LabelEncoder

In [4]:
le = LabelEncoder()

I need to fit and transform 'interest_level' in the training data

In [5]:
train_df['interest_level'] = le.fit_transform(train_df['interest_level'])

print("\n'interest_level' mapping (numerical to original):")
for i, label in enumerate(le.classes_):
    print(f"{i}: {label}")


'interest_level' mapping (numerical to original):
0: high
1: low
2: medium


im creating the features: a list of required amenities

In [6]:
amenities_to_extract = [
    'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman',
    'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War',
    'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom',
    'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'NewConstruction', 'Terrace',
    'Exclusive'
]

normalized names

In [7]:
normalized_amenities_set = {
    amenity.lower().replace(" ", "").replace("-", "")
    for amenity in amenities_to_extract
}

function to extract binary features for a given dataframe

In [8]:
def extract_amenity_features(df, amenities_set):
    df['features'] = df['features'].apply(lambda x: [str(s) for s in x] if isinstance(x, list) else [])

    for amenity in amenities_set:
        df[amenity.replace(" ", "").replace("-", "")] = df['features'].apply(
            lambda feature_list: 1 if amenity in [f.lower().replace(" ", "").replace("-", "") for f in feature_list] else 0
        )
    return df


applying function to both dataframes

In [9]:
train_df = extract_amenity_features(train_df.copy(), normalized_amenities_set)
test_df = extract_amenity_features(test_df.copy(), normalized_amenities_set)


print("\nTrain DataFrame after preprocessing and feature creation:")
print(train_df.head())
print("\nTest DataFrame after preprocessing and feature creation:")
print(test_df.head())

print("\nNew Train shape:", train_df.shape)
print("New Test shape:", test_df.shape)

# Verify one of the new columns
print(f"\nValue counts for 'elevator' in train_df:\n{train_df['elevator'].value_counts()}")



Train DataFrame after preprocessing and feature creation:
    bathrooms  bedrooms                       building_id  \
4         1.0         1  8579a0b0d54db803821a35a4a615e97a   
6         1.0         2  b8e75fc949a6cd8225b455648a951712   
9         1.0         2  cd759a988b8f23924b5a2058d5ab2b49   
10        1.5         3  53a5b119ba8f7b61d4e010512e0dfc85   
15        1.0         0  bfb9405149bfff42a92980b594c28234   

                created                                        description  \
4   2016-06-16 05:55:27  Spacious 1 Bedroom 1 Bathroom in Williamsburg!...   
6   2016-06-01 05:44:33  BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...   
9   2016-06-14 15:19:59  **FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...   
10  2016-06-24 07:54:24  A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...   
15  2016-06-28 03:50:23  Over-sized Studio w abundant closets. Availabl...   

        display_address                                           features  \
4   145 Borinquen Place  [D

## Task 3

im gonna convert 'created' column to datatime objects for date-based splitting

In [10]:
train_df['created'] = pd.to_datetime(train_df['created'])
test_df['created'] = pd.to_datetime(test_df['created'])

i split data into 2 parts (train and test) randomly.

In [11]:
def random_split_2_parts(df, test_size=0.2, random_state=42):
    train_sample, test_sample = train_test_split(df, test_size=test_size, random_state=random_state)
    print(f"\nRandom 2-part split: Train shape {train_sample.shape}, Test shape {test_sample.shape}")
    return train_sample, test_sample


i split data into 3 parts (train and test) randomly.

In [12]:
def random_split_3_parts(df, validation_size=0.2, test_size=0.2, random_state=42):
    if validation_size + test_size >= 1.0:
        raise ValueError("Sum of validation_size and test_size must be less than 1.0")

    # First, split into train_val and test
    train_val_sample, test_sample = train_test_split(df, test_size=test_size, random_state=random_state)

    # Calculate the new validation size relative to the train_val_sample
    relative_validation_size = validation_size / (1.0 - test_size)

    # Then, split train_val into train and validation
    train_sample, validation_sample = train_test_split(
        train_val_sample, test_size=relative_validation_size, random_state=random_state
    )
    print(f"\nRandom 3-part split: Train shape {train_sample.shape}, Validation shape {validation_sample.shape}, Test shape {test_sample.shape}")
    return train_sample, validation_sample, test_sample


now i split the data into 2 parts based on a specific date

In [13]:
def date_split_2_parts(df, date_split_point):
    if 'created' not in df.columns:
        raise ValueError("DataFrame must contain a 'created' column for date splitting.")
    if not isinstance(date_split_point, pd.Timestamp):
        date_split_point = pd.to_datetime(date_split_point)

    train_sample = df[df['created'] <= date_split_point].copy()
    test_sample = df[df['created'] > date_split_point].copy()
    print(f"\nDate 2-part split (split point: {date_split_point.date()}): Train shape {train_sample.shape}, Test shape {test_sample.shape}")
    return train_sample, test_sample

i split the data into 3 parts based on a specific date

In [14]:
def date_split_3_parts(df, validation_date, test_date):
    if 'created' not in df.columns:
        raise ValueError("DataFrame must contain a 'created' column for date splitting.")
    if not isinstance(validation_date, pd.Timestamp):
        validation_date = pd.to_datetime(validation_date)
    if not isinstance(test_date, pd.Timestamp):
        test_date = pd.to_datetime(test_date)

    if validation_date >= test_date:
        raise ValueError("validation_date must be strictly before test_date.")

    train_sample = df[df['created'] <= validation_date].copy()
    validation_sample = df[(df['created'] > validation_date) & (df['created'] <= test_date)].copy()
    test_sample = df[df['created'] > test_date].copy()

    print(f"\nDate 3-part split (validation date: {validation_date.date()}, test date: {test_date.date()}):")
    print(f"Train shape {train_sample.shape}, Validation shape {validation_sample.shape}, Test shape {test_sample.shape}")
    return train_sample, validation_sample, test_sample


## Task 4

Now im implementing the cross-validation methods

K-Fold cross-validation:

In [15]:
def k_fold_split(df, k=5, random_state=42, shuffle=True):
    kf = KFold(n_splits=k, shuffle=shuffle, random_state=random_state)
    splits = []
    print(f"\n--- K-Fold Cross-Validation (k={k}) ---")
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(df)):
        splits.append((train_idx, test_idx))
        print(f"Fold {fold_idx + 1}: Train indices count {len(train_idx)}, Test indices count {len(test_idx)}")
    return splits


Grouped K-Fold cross-validation:

In [16]:
def grouped_k_fold_split(df, k=5, group_field='manager_id'):
    if group_field not in df.columns:
        raise ValueError(f"DataFrame must contain a '{group_field}' column for Grouped K-Fold.")

    gkf = GroupKFold(n_splits=k)
    splits = []
    print(f"\n--- Grouped K-Fold Cross-Validation (k={k}, group_field='{group_field}') ---")
    # gkf.split takes X, y, and groups. Here, X and y can be df itself, and groups is the series.
    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, y=None, groups=df[group_field])):
        splits.append((train_idx, test_idx))
        print(f"Fold {fold_idx + 1}: Train indices count {len(train_idx)}, Test indices count {len(test_idx)}")
    return splits


Stratified K-Fold cross-validation:

In [17]:
def stratified_k_fold_split(df, k=5, stratify_field='interest_level', random_state=42, shuffle=True):
    if stratify_field not in df.columns:
        raise ValueError(f"DataFrame must contain a '{stratify_field}' column for Stratified K-Fold.")

    skf = StratifiedKFold(n_splits=k, shuffle=shuffle, random_state=random_state)
    splits = []
    print(f"\n--- Stratified K-Fold Cross-Validation (k={k}, stratify_field='{stratify_field}') ---")
    # skf.split takes X and y. Here, X can be df itself, and y is the stratify_field series.
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df, df[stratify_field])):
        splits.append((train_idx, test_idx))
        print(f"Fold {fold_idx + 1}: Train indices count {len(train_idx)}, Test indices count {len(test_idx)}")
    return splits

Time Series Split cross-validation:

In [18]:
def time_series_split(df, k=5, date_field='created'):
    if date_field not in df.columns:
        raise ValueError(f"DataFrame must contain a '{date_field}' column for Time Series Split.")
    # Ensuring data is sorted by date_field for TimeSeriesSplit
    df_sorted = df.sort_values(by=date_field).reset_index(drop=True)

    tscv = TimeSeriesSplit(n_splits=k)
    splits = []
    print(f"\n--- Time Series Split Cross-Validation (k={k}, date_field='{date_field}') ---")
    # tscv.split takes X. Here, X can be df_sorted itself.
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(df_sorted)):
        splits.append((train_idx, test_idx))
        print(f"Fold {fold_idx + 1}: Train indices count {len(train_idx)}, Test indices count {len(test_idx)}")
    return splits

## Task 5

I am going to create a function for distribution comparison

it prints descriptive statistics for numerical and value counts for categorical features


In [19]:
def print_feature_distributions(df_name, df_sample, numerical_cols, categorical_cols):
    print(f"\n--- Feature Distributions for {df_name} (Shape: {df_sample.shape}) ---")
    print("\nNumerical Features:")
    for col in numerical_cols:
        if col in df_sample.columns:
            print(f"  {col}: Mean={df_sample[col].mean():.4f}, Std={df_sample[col].std():.4f}")
        else:
            print(f"  {col}: Not found in DataFrame.")
    print("\nCategorical/Binary Features:")
    for col in categorical_cols:
        if col in df_sample.columns:
            print(f"  {col}:\n{df_sample[col].value_counts(normalize=True)}")
        else:
            print(f"  {col}: Not found in DataFrame.")


Im defining the features for comparison

In [20]:
numerical_features_for_comparison = ['bathrooms', 'bedrooms', 'price', 'latitude', 'longitude']
categorical_features_for_comparison = [
    'interest_level', 'elevator', 'hardwoodfloors', 'catsallowed', 'dogsallowed', 'doorman',
    'dishwasher', 'nofee', 'laundryinbuilding', 'fitnesscenter', 'prewar',
    'laundryinunit', 'roofdeck', 'outdoorspace', 'diningroom',
    'highspeedinternet', 'balcony', 'swimmingpool', 'newconstruction', 'terrace', 'exclusive'
]
print("\n" + "="*50)
print("--- Applying and Comparing Validation Methods ---")
print("="*50)



--- Applying and Comparing Validation Methods ---


random 2-part split

In [21]:
train_rand2, test_rand2 = random_split_2_parts(train_df)
print_feature_distributions("Random 2-part Train", train_rand2, numerical_features_for_comparison, categorical_features_for_comparison)



Random 2-part split: Train shape (39481, 35), Test shape (9871, 35)

--- Feature Distributions for Random 2-part Train (Shape: (39481, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2128, Std=0.5025
  bedrooms: Mean=1.5385, Std=1.1188
  price: Mean=3751.9920, Std=9861.9628
  latitude: Mean=40.7390, Std=0.7131
  longitude: Mean=-73.9511, Std=1.3151

Categorical/Binary Features:
  interest_level:
interest_level
1    0.695170
2    0.226463
0    0.078367
Name: proportion, dtype: float64
  elevator:
elevator
1    0.533041
0    0.466959
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.523594
1    0.476406
Name: proportion, dtype: float64
  catsallowed:
catsallowed
0    0.52258
1    0.47742
Name: proportion, dtype: float64
  dogsallowed:
dogsallowed
0    0.553481
1    0.446519
Name: proportion, dtype: float64
  doorman:
doorman
0    0.57415
1    0.42585
Name: proportion, dtype: float64
  dishwasher:
dishwasher
0    0.579241
1    0.420759
Name: proportion, dtype: fl

random 3-part split

In [22]:
train_rand3, val_rand3, test_rand3 = random_split_3_parts(train_df)
print_feature_distributions("Random 3-part Train", train_rand3, numerical_features_for_comparison, categorical_features_for_comparison)
print_feature_distributions("Random 3-part Validation", val_rand3, numerical_features_for_comparison, categorical_features_for_comparison)



Random 3-part split: Train shape (29610, 35), Validation shape (9871, 35), Test shape (9871, 35)

--- Feature Distributions for Random 3-part Train (Shape: (29610, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2099, Std=0.5011
  bedrooms: Mean=1.5315, Std=1.1185
  price: Mean=3767.9527, Std=11309.2079
  latitude: Mean=40.7360, Std=0.7878
  longitude: Mean=-73.9455, Std=1.4496

Categorical/Binary Features:
  interest_level:
interest_level
1    0.694394
2    0.225566
0    0.080041
Name: proportion, dtype: float64
  elevator:
elevator
1    0.5308
0    0.4692
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.523438
1    0.476562
Name: proportion, dtype: float64
  catsallowed:
catsallowed
0    0.522087
1    0.477913
Name: proportion, dtype: float64
  dogsallowed:
dogsallowed
0    0.552313
1    0.447687
Name: proportion, dtype: float64
  doorman:
doorman
0    0.577373
1    0.422627
Name: proportion, dtype: float64
  dishwasher:
dishwasher
0    0.580412
1    0.4195

In [23]:
date_show = pd.to_datetime('2016-05-15')
train_date2, test_date2 = date_split_2_parts(train_df, date_show)


Date 2-part split (split point: 2016-05-15): Train shape (24110, 35), Test shape (25242, 35)


Date 2-part split

In [24]:
min_date = train_df['created'].min()
max_date = train_df['created'].max()
mid_date_for_split = pd.to_datetime('2016-05-15')
if mid_date_for_split < min_date or mid_date_for_split > max_date:
    print(f"Warning: Chosen mid_date_for_split ({mid_date_for_split.date()}) is outside data range ({min_date.date()} to {max_date.date()}). Adjusting to middle.")
    mid_date_for_split = min_date + (max_date - min_date) / 2

train_date2, test_date2 = date_split_2_parts(train_df, mid_date_for_split)
print_feature_distributions("Date 2-part Train", train_date2, numerical_features_for_comparison, categorical_features_for_comparison)




Date 2-part split (split point: 2016-05-15): Train shape (24110, 35), Test shape (25242, 35)

--- Feature Distributions for Date 2-part Train (Shape: (24110, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2084, Std=0.4941
  bedrooms: Mean=1.5213, Std=1.1086
  price: Mean=3685.2288, Std=7355.8206
  latitude: Mean=40.7386, Std=0.6959
  longitude: Mean=-73.9520, Std=1.2668

Categorical/Binary Features:
  interest_level:
interest_level
1    0.685068
2    0.235048
0    0.079884
Name: proportion, dtype: float64
  elevator:
elevator
1    0.540689
0    0.459311
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.517047
1    0.482953
Name: proportion, dtype: float64
  catsallowed:
catsallowed
0    0.527375
1    0.472625
Name: proportion, dtype: float64
  dogsallowed:
dogsallowed
0    0.557072
1    0.442928
Name: proportion, dtype: float64
  doorman:
doorman
0    0.569266
1    0.430734
Name: proportion, dtype: float64
  dishwasher:
dishwasher
0    0.574285
1    0.425715


Date 3-part split

In [25]:
val_date_point = pd.to_datetime('2016-05-01')
test_date_point = pd.to_datetime('2016-06-01')

if val_date_point < min_date or test_date_point > max_date or val_date_point >= test_date_point:
    print("Warning: Chosen 3-part dates are problematic. Adjusting to approximate thirds.")
    val_date_point = min_date + (max_date - min_date) / 3
    test_date_point = min_date + 2 * (max_date - min_date) / 3

train_date3, val_date3, test_date3 = date_split_3_parts(train_df, val_date_point, test_date_point)
print_feature_distributions("Date 3-part Train", train_date3, numerical_features_for_comparison, categorical_features_for_comparison)
print_feature_distributions("Date 3-part Validation", val_date3, numerical_features_for_comparison, categorical_features_for_comparison)



Date 3-part split (validation date: 2016-05-01, test date: 2016-06-01):
Train shape (16411, 35), Validation shape (15797, 35), Test shape (17144, 35)

--- Feature Distributions for Date 3-part Train (Shape: (16411, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2078, Std=0.4935
  bedrooms: Mean=1.5178, Std=1.1060
  price: Mean=3646.9788, Std=2741.5007
  latitude: Mean=40.7448, Std=0.4522
  longitude: Mean=-73.9644, Std=0.8238

Categorical/Binary Features:
  interest_level:
interest_level
1    0.692219
2    0.230151
0    0.077631
Name: proportion, dtype: float64
  elevator:
elevator
1    0.53257
0    0.46743
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.514959
1    0.485041
Name: proportion, dtype: float64
  catsallowed:
catsallowed
0    0.530254
1    0.469746
Name: proportion, dtype: float64
  dogsallowed:
dogsallowed
0    0.560173
1    0.439827
Name: proportion, dtype: float64
  doorman:
doorman
0    0.573701
1    0.426299
Name: proportion, dtype: float6

Applying Cross-Validation Functions

K-fold

In [26]:
kf_splits = k_fold_split(train_df, k=5)
# Using the first fold's training data for comparison
train_kf_fold1 = train_df.iloc[kf_splits[0][0]]
print_feature_distributions("K-Fold (Fold 1) Train", train_kf_fold1, numerical_features_for_comparison, categorical_features_for_comparison)



--- K-Fold Cross-Validation (k=5) ---
Fold 1: Train indices count 39481, Test indices count 9871
Fold 2: Train indices count 39481, Test indices count 9871
Fold 3: Train indices count 39482, Test indices count 9870
Fold 4: Train indices count 39482, Test indices count 9870
Fold 5: Train indices count 39482, Test indices count 9870

--- Feature Distributions for K-Fold (Fold 1) Train (Shape: (39481, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2128, Std=0.5025
  bedrooms: Mean=1.5385, Std=1.1188
  price: Mean=3751.9920, Std=9861.9628
  latitude: Mean=40.7390, Std=0.7131
  longitude: Mean=-73.9511, Std=1.3151

Categorical/Binary Features:
  interest_level:
interest_level
1    0.695170
2    0.226463
0    0.078367
Name: proportion, dtype: float64
  elevator:
elevator
1    0.533041
0    0.466959
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.523594
1    0.476406
Name: proportion, dtype: float64
  catsallowed:
catsallowed
0    0.52258
1    0.47742
Name: propor

Grouped K-Fold

In [27]:
if 'manager_id' in train_df.columns:
    gkf_splits = grouped_k_fold_split(train_df, k=5, group_field='manager_id')
    train_gkf_fold1 = train_df.iloc[gkf_splits[0][0]]
    print_feature_distributions("Grouped K-Fold (Fold 1) Train", train_gkf_fold1, numerical_features_for_comparison, categorical_features_for_comparison)
else:
    print("\nSkipping Grouped K-Fold: 'manager_id' column not found in train_df.")




--- Grouped K-Fold Cross-Validation (k=5, group_field='manager_id') ---
Fold 1: Train indices count 39481, Test indices count 9871
Fold 2: Train indices count 39481, Test indices count 9871
Fold 3: Train indices count 39482, Test indices count 9870
Fold 4: Train indices count 39482, Test indices count 9870
Fold 5: Train indices count 39482, Test indices count 9870

--- Feature Distributions for Grouped K-Fold (Fold 1) Train (Shape: (39481, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2208, Std=0.5144
  bedrooms: Mean=1.5484, Std=1.1154
  price: Mean=3881.6325, Std=24045.0383
  latitude: Mean=40.7427, Std=0.6184
  longitude: Mean=-73.9568, Std=1.1420

Categorical/Binary Features:
  interest_level:
interest_level
1    0.698387
2    0.224614
0    0.076999
Name: proportion, dtype: float64
  elevator:
elevator
1    0.538512
0    0.461488
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.515488
1    0.484512
Name: proportion, dtype: float64
  catsallowed:
catsall

Stratified k-fold

In [28]:
skf_splits = stratified_k_fold_split(train_df, k=5, stratify_field='interest_level')
train_skf_fold1 = train_df.iloc[skf_splits[0][0]]
print_feature_distributions("Stratified K-Fold (Fold 1) Train", train_skf_fold1, numerical_features_for_comparison, categorical_features_for_comparison)



--- Stratified K-Fold Cross-Validation (k=5, stratify_field='interest_level') ---
Fold 1: Train indices count 39481, Test indices count 9871
Fold 2: Train indices count 39481, Test indices count 9871
Fold 3: Train indices count 39482, Test indices count 9870
Fold 4: Train indices count 39482, Test indices count 9870
Fold 5: Train indices count 39482, Test indices count 9870

--- Feature Distributions for Stratified K-Fold (Fold 1) Train (Shape: (39481, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2121, Std=0.5021
  bedrooms: Mean=1.5418, Std=1.1148
  price: Mean=3730.2274, Std=8282.7669
  latitude: Mean=40.7422, Std=0.6189
  longitude: Mean=-73.9572, Std=1.1482

Categorical/Binary Features:
  interest_level:
interest_level
1    0.694689
2    0.227527
0    0.077784
Name: proportion, dtype: float64
  elevator:
elevator
1    0.533978
0    0.466022
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.522378
1    0.477622
Name: proportion, dtype: float64
  catsallo

Time Series Split

In [29]:
tscv_splits = time_series_split(train_df, k=5, date_field='created')
df_sorted_for_tscv = train_df.sort_values(by='created').reset_index(drop=True)
train_tscv_fold1 = df_sorted_for_tscv.iloc[tscv_splits[0][0]]
print_feature_distributions("Time Series Split (Fold 1) Train", train_tscv_fold1, numerical_features_for_comparison, categorical_features_for_comparison)



--- Time Series Split Cross-Validation (k=5, date_field='created') ---
Fold 1: Train indices count 8227, Test indices count 8225
Fold 2: Train indices count 16452, Test indices count 8225
Fold 3: Train indices count 24677, Test indices count 8225
Fold 4: Train indices count 32902, Test indices count 8225
Fold 5: Train indices count 41127, Test indices count 8225

--- Feature Distributions for Time Series Split (Fold 1) Train (Shape: (8227, 35)) ---

Numerical Features:
  bathrooms: Mean=1.2057, Std=0.4986
  bedrooms: Mean=1.5077, Std=1.0988
  price: Mean=3634.2920, Std=3063.4121
  latitude: Mean=40.7498, Std=0.0460
  longitude: Mean=-73.9725, Std=0.0511

Categorical/Binary Features:
  interest_level:
interest_level
1    0.692962
2    0.224748
0    0.082290
Name: proportion, dtype: float64
  elevator:
elevator
1    0.52583
0    0.47417
Name: proportion, dtype: float64
  hardwoodfloors:
hardwoodfloors
0    0.512459
1    0.487541
Name: proportion, dtype: float64
  catsallowed:
catsallowe

Comparison and Best Choice:

* random splits(2-part and 3-part)

Both my random_split_2_parts and random_split_3_parts functions (which use sklearn.model_selection.train_test_split) produce training sets whose numerical feature means/standard deviations and categorical/binary feature proportions are very similar to the original full dataset.

* k-fold

My k_fold_split function (using sklearn.model_selection.KFold with shuffle=True) generates training folds that, on average, maintain similar feature distributions to the overall dataset. Each fold's training set is a large random subset.

* grouped k-fold

 My grouped_k_fold_split function (using sklearn.model_selection.GroupKFold) ensures that all data points belonging to a specific manager_id (or chosen group_field) are entirely within either the training or test set of a given fold. While this might lead to slight variations in overall feature distributions between folds compared to purely random splits (due to potentially uneven group sizes), the primary goal is to prevent group-level data leakage. The distributions within a given training fold will still be representative of the data excluding the test group.

* stratified k-fold

My stratified_k_fold_split function (using sklearn.model_selection.StratifiedKFold) explicitly ensures that the proportion of the target variable (interest_level) is preserved across all training and test folds. This is evident in the interest_level value counts being very consistent across the training folds. Other feature distributions also tend to remain stable due to this stratification

* time series split

My time_series_split function (using sklearn.model_selection.TimeSeriesSplit) sorts the data by created date and creates training sets from earlier periods and test sets from later periods. Consequently, the feature distributions in the training sets (especially the earlier folds) will reflect the characteristics of the data from those earlier time periods. These distributions might naturally differ from those in later test periods if the underlying data trends change over time.

comparison

* random splits:

pros: simpliest to implement and computionally inexpensive. good for large datasets

cons: could cause the leakage

* data splits:

pros: essential for time-series data and it strictly preserves the temporal order

cons: can result in training sets that are not representative of the overall data distribution if trends change significantly over time

* k-fold:
pros: provides a more robust and lefss biased estimate of model performance than a single train-test split by averaging results over multiple folds

cons: doesnt account for time dependencies or group dependencies

* grouped k-fold

pros:Crucial for datasets with inherent grouping and it guarantees that all samples from a single group are assigned to either the training or the test set within any given fold, effectively preventing group-level data leakage

cons: can lead to folds with unequal sizes if group sizes vary widely, potentially affecting the stability of performance estimates

* stratified k-fold

pros: highly recommended for classification problems, especially when the target variable is imbalanced. It ensures that each fold (both training and test) maintains approximately the same proportion of target classes as the original dataset. This leads to more reliable and less biased performance metrics.

cons: does not inherently account for time dependencies or group dependencies

* time series split

pros: the definitive method for time-series forecasting. It strictly adheres to the temporal order, always training on past data and evaluating on future data. This is the most realistic simulation of a model's performance in a production environment for time-dependent problems.

cons: the training set size grows with each fold, meaning earlier folds train on less data. It does not shuffle the data, which is appropriate for time series but means it won't mix data from different periods.

The best scheme for the dataset:

For our current dataset, cross-validation techniques would not differ a lot. Usual K-fold will suit our aims since the data doesn't have strong time dependencies that would significantly affect the model. However, if we have a long tail in the target variable distribution (e.g., expensive houses), it would be better to use Stratified K-fold and split expensive houses preserving the train and test ratio. To do that, we would need to bin the continuous target variable into discrete categories first.

## Task 6

In [30]:
model_features = [
    'bathrooms', 'bedrooms',
    'elevator', 'hardwoodfloors', 'catsallowed', 'dogsallowed', 'doorman',
    'dishwasher', 'nofee', 'laundryinbuilding', 'fitnesscenter', 'prewar',
    'laundryinunit', 'roofdeck', 'outdoorspace', 'diningroom',
    'highspeedinternet', 'balcony', 'swimmingpool', 'newconstruction', 'terrace',
    'exclusive'
]

target_feature = 'price'

im filtering out features that might not exist in the data frame

In [31]:
available_model_features = [f for f in model_features if f in train_df.columns]
print(f"\nFeatures available for modeling: {len(available_model_features)} out of {len(model_features)}")
print(available_model_features)


Features available for modeling: 22 out of 22
['bathrooms', 'bedrooms', 'elevator', 'hardwoodfloors', 'catsallowed', 'dogsallowed', 'doorman', 'dishwasher', 'nofee', 'laundryinbuilding', 'fitnesscenter', 'prewar', 'laundryinunit', 'roofdeck', 'outdoorspace', 'diningroom', 'highspeedinternet', 'balcony', 'swimmingpool', 'newconstruction', 'terrace', 'exclusive']


creating a function to fit and evaluate a logistic regression model

In [32]:
def fit_and_evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, scaler=None, model_name="Model"):
    """
    Fits and evaluates a regression model on train, validation, and test sets.
    Reports MAE, RMSE, and R² metrics.
    """
    start_time = time.time()

    if scaler:
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
    else:
        X_train_scaled = X_train
        X_val_scaled = X_val
        X_test_scaled = X_test

    model.fit(X_train_scaled, y_train)
    fit_time = time.time() - start_time

    y_train_pred = model.predict(X_train_scaled)
    y_val_pred = model.predict(X_val_scaled)
    y_test_pred = model.predict(X_test_scaled)

    train_mae = mean_absolute_error(y_train, y_train_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_r2 = r2_score(y_train, y_train_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"\n--- {model_name} Performance ---")
    print(f"Fit Time: {fit_time:.4f} seconds")
    print(f"MAE  - Train: {train_mae:.6f}, Valid: {val_mae:.6f}, Test: {test_mae:.6f}")
    print(f"RMSE - Train: {train_rmse:.6f}, Valid: {val_rmse:.6f}, Test: {test_rmse:.6f}")
    print(f"R²   - Train: {train_r2:.6f}, Valid: {val_r2:.6f}, Test: {test_r2:.6f}")

    return {
        'model': model,
        'fit_time': fit_time,
        'train_mae': train_mae, 'val_mae': val_mae, 'test_mae': test_mae,
        'train_rmse': train_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
        'train_r2': train_r2, 'val_r2': val_r2, 'test_r2': test_r2
    }


lasso Regression with 60/20/20 Date Split

In [33]:
pip show scikit-learn

Name: scikit-learn
Version: 1.8.0
Summary: A set of python modules for machine learning and data mining
Home-page: 
Author: 
Author-email: 
License: 
Location: /home/user01/ML3_Validation.ID_1254800-1/src/.venv/lib/python3.12/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: shap
Note: you may need to restart the kernel to use updated packages.


In [34]:
# Split data by date with 60/20/20 ratio
min_date_data = train_df['created'].min()
max_date_data = train_df['created'].max()
total_days = (max_date_data - min_date_data).days

val_split_date = min_date_data + pd.Timedelta(days=int(total_days * 0.60))
test_split_date = min_date_data + pd.Timedelta(days=int(total_days * 0.80))

print(f"Calculated split dates: Validation before {val_split_date.date()}, Test before {test_split_date.date()}")

train_lasso_df, val_lasso_df, test_lasso_df = date_split_3_parts(
    train_df, validation_date=val_split_date, test_date=test_split_date
)

X_train_lasso = train_lasso_df[available_model_features]
y_train_lasso = train_lasso_df[target_feature]
X_val_lasso = val_lasso_df[available_model_features]
y_val_lasso = val_lasso_df[target_feature]
X_test_lasso = test_lasso_df[available_model_features]
y_test_lasso = test_lasso_df[target_feature]

# Initialize MinMaxScaler
scaler_lasso = MinMaxScaler()

# Lasso regression model
initial_lasso_model = Lasso(alpha=1.0, random_state=42, max_iter=10000)

print("\n--- Initial Lasso Model (All Features) ---")
initial_lasso_results = fit_and_evaluate_model(
    initial_lasso_model, X_train_lasso, y_train_lasso, X_val_lasso, y_val_lasso, X_test_lasso, y_test_lasso,
    scaler=scaler_lasso, model_name="Lasso MinMaxScaler"
)

# Sort features by absolute weight coefficients
if hasattr(initial_lasso_model, 'coef_'):
    feature_importances = pd.DataFrame({
        'Name': available_model_features,
        'Importance': np.abs(initial_lasso_model.coef_)
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    feature_importances.index.name = 'ID'
    print(f"\nFeature Importances (by absolute Lasso coefficients):")
    print(feature_importances.to_string())

    top_10_lasso_features = feature_importances.head(10)['Name'].tolist()
    print(f"\nTop 10 Features: {top_10_lasso_features}")

    # Refit model to top 10 features
    scaler_lasso_top10 = MinMaxScaler()
    refitted_lasso_model = Lasso(alpha=1.0, random_state=42, max_iter=10000)

    print("\n--- Lasso Model (Top 10 Features by Coefficients) ---")
    lasso_top10_results = fit_and_evaluate_model(
        refitted_lasso_model,
        train_lasso_df[top_10_lasso_features], y_train_lasso,
        val_lasso_df[top_10_lasso_features], y_val_lasso,
        test_lasso_df[top_10_lasso_features], y_test_lasso,
        scaler=scaler_lasso_top10, model_name="Lasso top10 MinMaxScaler"
    )
else:
    print("Lasso model did not have coefficients.")
    top_10_lasso_features = []
    lasso_top10_results = {}


Calculated split dates: Validation before 2016-05-23, Test before 2016-06-10

Date 3-part split (validation date: 2016-05-23, test date: 2016-06-10):
Train shape (28918, 35), Validation shape (9476, 35), Test shape (10958, 35)

--- Initial Lasso Model (All Features) ---

--- Lasso MinMaxScaler Performance ---
Fit Time: 0.0236 seconds
MAE  - Train: 975.216936, Valid: 912.372532, Test: 1411.863700
RMSE - Train: 9079.175277, Valid: 1998.999007, Test: 44253.600733
R²   - Train: 0.032636, Valid: 0.439556, Test: 0.001757

Feature Importances (by absolute Lasso coefficients):
                 Name    Importance
ID                                 
0           bathrooms  23698.997403
1            bedrooms   2828.112955
2             doorman   1171.970075
3             terrace    535.804271
4       laundryinunit    528.611180
5   laundryinbuilding    421.075757
6         dogsallowed    327.843929
7   highspeedinternet    301.724656
8               nofee    285.582001
9             balcony    217

 Feature Selection by NaN Ratio and Correlation

In [35]:
def select_features_by_nan_and_corr(df, target_col, feature_cols, nan_threshold=0.5, corr_threshold=0.01, top_n=10):
    """
    Selects features based on NaN ratio and absolute Pearson correlation with the target.
    """
    print("\n--- Feature Selection by NaN Ratio and Correlation ---")

    features_after_nan_filter = []
    for col in feature_cols:
        if col in df.columns:
            nan_ratio = df[col].isnull().sum() / len(df)
            if nan_ratio < nan_threshold:
                features_after_nan_filter.append(col)
            else:
                print(f"  Skipping '{col}' due to high NaN ratio ({nan_ratio:.2f})")
        else:
            print(f"  Warning: Feature '{col}' not found in DataFrame.")

    correlations = {}
    for col in features_after_nan_filter:
        if df[col].dtype in ['int64', 'float64']:
            corr_val = df[col].corr(df[target_col])
            correlations[col] = abs(corr_val)

    sorted_correlations = sorted(correlations.items(), key=lambda item: item[1], reverse=True)
    top_features = [feat for feat, corr in sorted_correlations if corr >= corr_threshold][:top_n]

    print(f"  Top {top_n} features by correlation (abs >= {corr_threshold}): {top_features}")
    return top_features

print("\n--- Applying Feature Selection by NaN Ratio and Correlation ---")
top_10_nan_corr_features = select_features_by_nan_and_corr(
    train_lasso_df, target_feature, available_model_features
)

if top_10_nan_corr_features:
    scaler_nan_corr_top10 = MinMaxScaler()
    nan_corr_model = Lasso(alpha=1.0, random_state=42, max_iter=10000)

    print("\n--- Lasso Model (Top 10 Features by NaN/Correlation) ---")
    nan_corr_results = fit_and_evaluate_model(
        nan_corr_model,
        train_lasso_df[top_10_nan_corr_features], y_train_lasso,
        val_lasso_df[top_10_nan_corr_features], y_val_lasso,
        test_lasso_df[top_10_nan_corr_features], y_test_lasso,
        scaler=scaler_nan_corr_top10, model_name="Lasso NaN/Corr MinMaxScaler"
    )
else:
    print("No features selected by NaN Ratio and Correlation method.")
    nan_corr_results = {}


--- Applying Feature Selection by NaN Ratio and Correlation ---

--- Feature Selection by NaN Ratio and Correlation ---
  Top 10 features by correlation (abs >= 0.01): ['bathrooms', 'bedrooms', 'doorman', 'laundryinunit', 'diningroom', 'elevator', 'fitnesscenter', 'dishwasher', 'terrace', 'outdoorspace']

--- Lasso Model (Top 10 Features by NaN/Correlation) ---

--- Lasso NaN/Corr MinMaxScaler Performance ---
Fit Time: 0.0153 seconds
MAE  - Train: 972.870037, Valid: 913.781681, Test: 1410.262012
RMSE - Train: 9084.172575, Valid: 2005.110045, Test: 44258.966596
R²   - Train: 0.031571, Valid: 0.436124, Test: 0.001515


Permutation Importance

In [36]:
def select_features_by_permutation_importance(model, X, y, feature_names, scoring='neg_mean_absolute_error', top_n=10, scaler=None):
    """
    Calculates permutation importance and returns top N features.
    """
    print("\n--- Feature Selection by Permutation Importance ---")
    start_time = time.time()

    if scaler:
        X_scaled = scaler.transform(X)
    else:
        X_scaled = X

    result = permutation_importance(model, X_scaled, y, n_repeats=5, random_state=42, scoring=scoring, n_jobs=-1)

    importances = pd.Series(result.importances_mean, index=feature_names)
    # For neg_mean_absolute_error, more negative = more important (bigger drop)
    # Sort by absolute value descending
    importances_sorted = importances.abs().sort_values(ascending=False)
    
    # Also show mean ± std
    print("\nneg_mean_absolute_percentage_error:")
    print(f"{'Feature':<25} {'Mean ± Std'}")
    for feat in importances_sorted.head(top_n).index:
        feat_idx = list(feature_names).index(feat)
        mean_val = abs(result.importances_mean[feat_idx])
        std_val = result.importances_std[feat_idx]
        print(f"{feat:<25} {mean_val:.5f} ± {std_val:.3f}")
    
    top_features = importances_sorted.head(top_n).index.tolist()

    end_time = time.time()
    print(f"\n  Permutation Importance calculation time: {end_time - start_time:.4f} seconds")
    print(f"  Top {top_n} features by Permutation Importance: {top_features}")
    return top_features

print("\n--- Applying Permutation Importance ---")

if initial_lasso_model and hasattr(initial_lasso_model, 'coef_'):
    top_10_perm_imp_features = select_features_by_permutation_importance(
        initial_lasso_model, X_val_lasso, y_val_lasso, available_model_features,
        scoring='neg_mean_absolute_error',
        scaler=scaler_lasso
    )

    if top_10_perm_imp_features:
        scaler_perm_imp_top10 = MinMaxScaler()
        perm_imp_model = Lasso(alpha=1.0, random_state=42, max_iter=10000)

        print("\n--- Lasso Model (Top 10 Features by Permutation Importance) ---")
        perm_imp_results = fit_and_evaluate_model(
            perm_imp_model,
            train_lasso_df[top_10_perm_imp_features], y_train_lasso,
            val_lasso_df[top_10_perm_imp_features], y_val_lasso,
            test_lasso_df[top_10_perm_imp_features], y_test_lasso,
            scaler=scaler_perm_imp_top10, model_name="Lasso permutation MinMaxScaler"
        )
    else:
        print("No features selected by Permutation Importance.")
        perm_imp_results = {}
else:
    print("Initial Lasso model not available.")
    perm_imp_results = {}


--- Applying Permutation Importance ---

--- Feature Selection by Permutation Importance ---

neg_mean_absolute_percentage_error:
Feature                   Mean ± Std
bathrooms                 482.05535 ± 4.672
doorman                   214.27357 ± 2.100
bedrooms                  152.96496 ± 2.801
laundryinunit             34.59194 ± 1.634
laundryinbuilding         23.24200 ± 1.842
dogsallowed               10.95850 ± 0.910
nofee                     9.26440 ± 0.478
catsallowed               8.48559 ± 0.565
terrace                   6.32324 ± 0.643
highspeedinternet         5.13001 ± 0.427

  Permutation Importance calculation time: 2.4003 seconds
  Top 10 features by Permutation Importance: ['bathrooms', 'doorman', 'bedrooms', 'laundryinunit', 'laundryinbuilding', 'dogsallowed', 'nofee', 'catsallowed', 'terrace', 'highspeedinternet']

--- Lasso Model (Top 10 Features by Permutation Importance) ---

--- Lasso permutation MinMaxScaler Performance ---
Fit Time: 0.0102 seconds
MAE  - Trai

SHAP importance

In [37]:
def select_features_by_shap(model, X, feature_names, top_n=10, scaler=None):
    """
    Calculates SHAP values and returns top N features based on mean absolute SHAP value.
    """
    print("\n--- Feature Selection by SHAP Importance ---")
    start_time = time.time()

    if scaler:
        X_scaled = scaler.transform(X)
    else:
        X_scaled = X

    # For Lasso (linear model), use LinearExplainer
    explainer = shap.LinearExplainer(model, X_scaled)
    shap_values = explainer.shap_values(X_scaled)

    # Calculate mean absolute SHAP value for each feature
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

    shap_importances = pd.DataFrame({
        'feature': feature_names,
        'shap_value': mean_abs_shap
    }).sort_values('shap_value', ascending=False).reset_index(drop=True)
    shap_importances.index.name = 'id'

    print(f"\nSHAP Feature Importances:")
    print(shap_importances.to_string())

    top_features = shap_importances.head(top_n)['feature'].tolist()

    end_time = time.time()
    print(f"\n  SHAP calculation time: {end_time - start_time:.4f} seconds")
    print(f"  Top {top_n} features by SHAP: {top_features}")
    return top_features

print("\n--- Applying SHAP Importance ---")
if initial_lasso_model and hasattr(initial_lasso_model, 'coef_'):
    top_10_shap_features = select_features_by_shap(
        initial_lasso_model, X_val_lasso, available_model_features,
        scaler=scaler_lasso
    )

    if top_10_shap_features:
        scaler_shap_top10 = MinMaxScaler()
        shap_model = Lasso(alpha=1.0, random_state=42, max_iter=10000)

        print("\n--- Lasso Model (Top 10 Features by SHAP Importance) ---")
        shap_results = fit_and_evaluate_model(
            shap_model,
            train_lasso_df[top_10_shap_features], y_train_lasso,
            val_lasso_df[top_10_shap_features], y_val_lasso,
            test_lasso_df[top_10_shap_features], y_test_lasso,
            scaler=scaler_shap_top10, model_name="Lasso shap MinMaxScaler"
        )
    else:
        print("No features selected by SHAP.")
        shap_results = {}
else:
    print("Initial Lasso model not available.")
    shap_results = {}


--- Applying SHAP Importance ---

--- Feature Selection by SHAP Importance ---

SHAP Feature Importances:
              feature  shap_value
id                               
0           bathrooms  828.624590
1             doorman  593.069297
2            bedrooms  377.056714
3   laundryinbuilding  209.524737
4       laundryinunit  180.589109
5         dogsallowed  162.219778
6               nofee  129.467859
7      hardwoodfloors   82.133855
8         catsallowed   64.465213
9            elevator   61.347013
10  highspeedinternet   54.349921
11      fitnesscenter   50.645680
12         diningroom   41.770022
13            terrace   40.736052
14             prewar   38.080387
15            balcony   31.257685
16       outdoorspace   29.009670
17           roofdeck   14.948913
18         dishwasher    1.759527
19          exclusive    0.463760
20    newconstruction    0.000000
21       swimmingpool    0.000000

  SHAP calculation time: 0.0131 seconds
  Top 10 features by SHAP: ['bathroo

In [38]:
# Comparison summary of all feature selection methods
results_summary = {
    "Lasso MinMaxScaler": initial_lasso_results,
    "Lasso top10 MinMaxScaler": lasso_top10_results,
    "Lasso permutation MinMaxScaler": perm_imp_results,
    "Lasso shap MinMaxScaler": shap_results
}

# Print MAE table
print("\nMAE")
print(f"{'model':<35} {'train':<15} {'valid':<15} {'test':<15}")
for name, res in results_summary.items():
    if res:
        print(f"{name:<35} {res['train_mae']:<15.6f} {res['val_mae']:<15.6f} {res['test_mae']:<15.6f}")

# Print RMSE table
print("\nRMSE")
print(f"{'model':<35} {'train':<15} {'valid':<15} {'test':<15}")
for name, res in results_summary.items():
    if res:
        print(f"{name:<35} {res['train_rmse']:<15.6f} {res['val_rmse']:<15.6f} {res['test_rmse']:<15.6f}")

# Print R² table
print("\nR²")
print(f"{'model':<35} {'train':<15} {'valid':<15} {'test':<15}")
for name, res in results_summary.items():
    if res:
        print(f"{name:<35} {res['train_r2']:<15.6f} {res['val_r2']:<15.6f} {res['test_r2']:<15.6f}")

print("\n--- Qualitative Comparison ---")
print("\n**Speed of Feature Selection:**")
print("- **Lasso Coefficients:** Very fast, as it's inherent to the model fitting process.")
print("- **Permutation Importance:** Moderately slow, requires re-evaluating the model multiple times.")
print("- **SHAP:** Can be computationally intensive; LinearExplainer is faster for linear models.")

print("\n**Metrics and Stability:**")
print("- There is no significant difference in metrics compared to the model on all features.")
print("- All feature selection methods give approximately the same top features.")
print("- Lasso coefficients, permutation importance, and SHAP all identify bathrooms, bedrooms, and Doorman as top features.")



MAE
model                               train           valid           test           
Lasso MinMaxScaler                  975.216936      912.372532      1411.863700    
Lasso top10 MinMaxScaler            969.440908      909.947027      1407.544154    
Lasso permutation MinMaxScaler      968.346337      908.781248      1405.180540    
Lasso shap MinMaxScaler             968.923998      905.888913      1401.746085    

RMSE
model                               train           valid           test           
Lasso MinMaxScaler                  9079.175277     1998.999007     44253.600733   
Lasso top10 MinMaxScaler            9080.061509     2000.479693     44254.855162   
Lasso permutation MinMaxScaler      9080.196765     1998.823813     44254.927526   
Lasso shap MinMaxScaler             9080.710829     1997.556776     44254.334820   

R²
model                               train           valid           test           
Lasso MinMaxScaler                  0.032636        0.439556 

Comparison

**Speed comparison:**
* Lasso Coefficients: Very fast — feature importance is a direct outcome of the model fitting.
* NaN Ratio & Correlation: Generally fast — simple statistical calculations.
* Permutation Importance: Moderately slow — requires re-evaluating the model multiple times.
* SHAP: Can be computationally intensive; LinearExplainer is faster for linear models.

**Overall:** For interpretable linear models like Lasso, coefficient-based selection is straightforward. Permutation Importance and SHAP are highly recommended for their model-agnostic nature and robust importance estimations. There is no significant difference in metrics compared to model on all features.

## Task 7

Im gonna define the ElasticNet-regularized Logistic Regression model and parameter space

In [39]:
# Parameter grid for ElasticNet
param_grid_elasticnet = {
    'alpha': np.logspace(-3, 0, 4),  # e.g., 0.001, 0.01, 0.1, 1.0
    'l1_ratio': np.linspace(0.1, 1.0, 5)  # e.g., 0.1, 0.325, 0.55, 0.775, 1.0
}


training the data that hpo methods will use for their internal cv

In [40]:
X_hpo_train = X_train_lasso
y_hpo_train = y_train_lasso

In [41]:
scorer = make_scorer(mean_absolute_error, greater_is_better=False)


Grid Search for ElasticNet Logistic Regression

In [42]:
print("\n--- Grid Search for ElasticNet Regression ---")
start_time_grid = time.time()

grid_model = ElasticNet(random_state=42, max_iter=10000)
grid_search = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid_elasticnet,
    scoring='neg_mean_absolute_error',
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    verbose=1,
    n_jobs=-1
)

scaler_grid = MinMaxScaler()
X_hpo_train_scaled_grid = scaler_grid.fit_transform(X_hpo_train)

grid_search.fit(X_hpo_train_scaled_grid, y_hpo_train)
end_time_grid = time.time()
grid_search_time = end_time_grid - start_time_grid

print(f"\nGrid Search Best Parameters: {grid_search.best_params_}")
print(f"Grid Search Best MAE (negative): {grid_search.best_score_:.4f}")
print(f"Grid Search Time: {grid_search_time:.4f} seconds")

best_grid_model = grid_search.best_estimator_
grid_results = fit_and_evaluate_model(
    best_grid_model, X_train_lasso, y_train_lasso, X_val_lasso, y_val_lasso, X_test_lasso, y_test_lasso,
    scaler=scaler_grid,
    model_name="ElasticNet (Grid Search Best)"
)



--- Grid Search for ElasticNet Regression ---
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Grid Search Best Parameters: {'alpha': np.float64(0.001), 'l1_ratio': np.float64(0.55)}
Grid Search Best MAE (negative): -980.9567
Grid Search Time: 1.1482 seconds

--- ElasticNet (Grid Search Best) Performance ---
Fit Time: 0.0422 seconds
MAE  - Train: 971.874459, Valid: 908.841982, Test: 1412.277598
RMSE - Train: 9081.480901, Valid: 2018.124209, Test: 44251.419041
R²   - Train: 0.032144, Valid: 0.428781, Test: 0.001855


Random Search for ElasticNet Logistic Regression

In [43]:
print("\n--- Random Search for ElasticNet Regression ---")
start_time_random = time.time()

from scipy.stats import loguniform, uniform

param_distributions_elasticnet = {
    'alpha': loguniform(0.001, 1.0),
    'l1_ratio': uniform(0.1, 0.9)
}

random_model = ElasticNet(random_state=42, max_iter=10000)
random_search = RandomizedSearchCV(
    estimator=random_model,
    param_distributions=param_distributions_elasticnet,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    verbose=1,
    random_state=42,
    n_jobs=-1
)

scaler_random = MinMaxScaler()
X_hpo_train_scaled_random = scaler_random.fit_transform(X_hpo_train)

random_search.fit(X_hpo_train_scaled_random, y_hpo_train)
end_time_random = time.time()
random_search_time = end_time_random - start_time_random

print(f"\nRandom Search Best Parameters: {random_search.best_params_}")
print(f"Random Search Best MAE (negative): {random_search.best_score_:.4f}")
print(f"Random Search Time: {random_search_time:.4f} seconds")

best_random_model = random_search.best_estimator_
random_results = fit_and_evaluate_model(
    best_random_model, X_train_lasso, y_train_lasso, X_val_lasso, y_val_lasso, X_test_lasso, y_test_lasso,
    scaler=scaler_random,
    model_name="ElasticNet (Random Search Best)"
)


--- Random Search for ElasticNet Regression ---
Fitting 3 folds for each of 50 candidates, totalling 150 fits

Random Search Best Parameters: {'alpha': np.float64(0.0026471141828218157), 'l1_ratio': np.float64(0.8219772826786357)}
Random Search Best MAE (negative): -980.9690
Random Search Time: 2.5941 seconds

--- ElasticNet (Random Search Best) Performance ---
Fit Time: 0.0411 seconds
MAE  - Train: 971.929038, Valid: 908.913084, Test: 1412.474173
RMSE - Train: 9081.655717, Valid: 2019.244226, Test: 44251.370758
R²   - Train: 0.032107, Valid: 0.428146, Test: 0.001857


Optuna for ElasticNet Logistic Regression

In [44]:
print("\n--- Optuna for ElasticNet Regression ---")

def objective(trial):
    """
    Objective function for Optuna to optimize ElasticNet.
    Uses KFold for cross-validation internally.
    """
    alpha = trial.suggest_float('alpha', 0.001, 1.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.1, 1.0)

    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        random_state=42,
        max_iter=10000
    )

    cv = KFold(n_splits=3, shuffle=True, random_state=42)

    scaler_optuna_cv = MinMaxScaler()
    X_hpo_train_scaled_optuna = scaler_optuna_cv.fit_transform(X_hpo_train)

    scores = cross_val_score(
        model, X_hpo_train_scaled_optuna, y_hpo_train,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )

    # Optuna minimizes the objective
    return -np.mean(scores)  # Return positive MAE to minimize

start_time_optuna = time.time()
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)
end_time_optuna = time.time()
optuna_search_time = end_time_optuna - start_time_optuna

print(f"\nOptuna Best Parameters: {study.best_params}")
print(f"Optuna Best MAE: {study.best_value:.4f}")
print(f"Optuna Search Time: {optuna_search_time:.4f} seconds")

best_optuna_params = study.best_params
best_optuna_model = ElasticNet(
    alpha=best_optuna_params['alpha'],
    l1_ratio=best_optuna_params['l1_ratio'],
    random_state=42,
    max_iter=10000
)

scaler_optuna_best = MinMaxScaler()
optuna_results = fit_and_evaluate_model(
    best_optuna_model, X_train_lasso, y_train_lasso, X_val_lasso, y_val_lasso, X_test_lasso, y_test_lasso,
    scaler=scaler_optuna_best,
    model_name="ElasticNet (Optuna Best)"
)

print("\nOptuna should perform fewer operations to approach the optimal parameters.")



--- Optuna for ElasticNet Regression ---


  0%|          | 0/50 [00:00<?, ?it/s]


Optuna Best Parameters: {'alpha': 0.059746858603357024, 'l1_ratio': 0.9927502321813144}
Optuna Best MAE: 980.7759
Optuna Search Time: 8.0954 seconds

--- ElasticNet (Optuna Best) Performance ---
Fit Time: 0.0296 seconds
MAE  - Train: 971.673867, Valid: 908.637865, Test: 1411.973720
RMSE - Train: 9081.364898, Valid: 2017.332410, Test: 44251.481869
R²   - Train: 0.032169, Valid: 0.429229, Test: 0.001852

Optuna should perform fewer operations to approach the optimal parameters.


Summary

In [45]:
print("\n" + "="*50)
print("--- Comparison of Hyperparameter Optimization Approaches ---")
print("="*50)

hpo_results_summary = {
    "Grid Search": grid_results,
    "Random Search": random_results,
    "Optuna (Bayesian)": optuna_results
}

# MAE comparison
print("\nMAE")
print(f"{'Method':<35} {'train':<15} {'valid':<15} {'test':<15} {'Search Time (s)':<15}")
print("-" * 85)
for method_name, res in hpo_results_summary.items():
    if res:
        if method_name == "Grid Search":
            search_time = grid_search_time
        elif method_name == "Random Search":
            search_time = random_search_time
        else:
            search_time = optuna_search_time
        print(f"{method_name:<35} {res['train_mae']:<15.6f} {res['val_mae']:<15.6f} {res['test_mae']:<15.6f} {search_time:<15.4f}")

# RMSE comparison
print("\nRMSE")
print(f"{'Method':<35} {'train':<15} {'valid':<15} {'test':<15}")
print("-" * 85)
for method_name, res in hpo_results_summary.items():
    if res:
        print(f"{method_name:<35} {res['train_rmse']:<15.6f} {res['val_rmse']:<15.6f} {res['test_rmse']:<15.6f}")

# R² comparison
print("\nR²")
print(f"{'Method':<35} {'train':<15} {'valid':<15} {'test':<15}")
print("-" * 85)
for method_name, res in hpo_results_summary.items():
    if res:
        print(f"{method_name:<35} {res['train_r2']:<15.6f} {res['val_r2']:<15.6f} {res['test_r2']:<15.6f}")

print("\n--- Qualitative Comparison of HPO Approaches ---")
print("- Grid Search: Exhaustive but slow. Guaranteed to find the best within the grid.")
print("- Random Search: Faster, samples random combinations. Good for large search spaces.")
print("- Optuna (Bayesian): Most efficient. Uses past results to guide search. Finds good parameters in fewer trials.")
print("\nOptuna should perform fewer operations to approach the optimal parameters.")



--- Comparison of Hyperparameter Optimization Approaches ---

MAE
Method                              train           valid           test            Search Time (s)
-------------------------------------------------------------------------------------
Grid Search                         971.874459      908.841982      1412.277598     1.1482         
Random Search                       971.929038      908.913084      1412.474173     2.5941         
Optuna (Bayesian)                   971.673867      908.637865      1411.973720     8.0954         

RMSE
Method                              train           valid           test           
-------------------------------------------------------------------------------------
Grid Search                         9081.480901     2018.124209     44251.419041   
Random Search                       9081.655717     2019.244226     44251.370758   
Optuna (Bayesian)                   9081.364898     2017.332410     44251.481869   

R²
Method         